# 세션별 대화 기록 — 강사용 정답본

원본: `solution/Chapter01/02-langchain/04_memory_runnable.py`  
대응 슬라이드: `99_pdf/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf`

**학습 목표:** 히스토리 저장소 → 래퍼 → 세션 A/B 비교의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 세션별 질문에서 LLM을 호출한다. 대화 기록은 이 프로세스의 메모리에 쌓인다.



## 흐름을 먼저 읽기

히스토리 저장소 → 래퍼 → 세션 A/B 비교

핵심은 각 Runnable의 입력·출력과 반복 흐름이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 히스토리 저장소 → 래퍼 → 세션 A/B 비교. 같은 세션 ID를 재사용하면 어떤 기록이 보이나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 세션별 질문에서 LLM을 호출한다. 대화 기록은 이 프로세스의 메모리에 쌓인다.



In [ ]:
"""
메모리 전략 1 — LangChain 방식: RunnableWithMessageHistory

대화 기록을 체인 외부(InMemoryChatMessageHistory)에서 세션별로 관리.
체인 자체는 순수 함수처럼 동작하고, 히스토리는 wrapper가 주입한다.

실행:
  python 04_memory_runnable.py
"""

from dotenv import load_dotenv
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── 세션 저장소 ───────────────────────────────────────────────────────────────
# 실제 서비스에서는 Redis·DB로 교체 가능

session_store: dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]


# ── 체인 구성 ─────────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 친절한 AI 어시스턴트야.'),
    MessagesPlaceholder(variable_name='history'),  # 이전 대화가 여기 삽입됨
    ('human', '{input}'),
])

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history',
)


# ── 세션 A ────────────────────────────────────────────────────────────────────
print('=== 세션 A: 홍길동 ===')
config_a = {'configurable': {'session_id': 'session_A'}}

r1 = chain_with_history.invoke({'input': '내 이름은 홍길동이야.'}, config=config_a)
print(f'AI: {r1.content}')

r2 = chain_with_history.invoke({'input': '내 이름이 뭐야?'}, config=config_a)
print(f'AI: {r2.content}')


# ── 세션 B ────────────────────────────────────────────────────────────────────
print('\n=== 세션 B: 별개 대화 ===')
config_b = {'configurable': {'session_id': 'session_B'}}

r3 = chain_with_history.invoke({'input': '내 이름이 뭐야?'}, config=config_b)
print(f'AI: {r3.content}')  # B는 A 대화를 모름


# ── 저장된 히스토리 확인 ──────────────────────────────────────────────────────
print('\n=== 세션 A 저장된 메시지 ===')
for msg in get_session_history('session_A').messages:
    role = '사람' if msg.type == 'human' else ' AI '
    print(f'[{role}] {msg.content}')


# ── LangChain 방식의 한계 ─────────────────────────────────────────────────────
print('''
[LangChain 방식 한계]
- 히스토리가 체인 외부에 있어 그래프와 통합이 어려움
- 도구 실행 결과가 메시지 흐름에 자동 포함되지 않음
- 복잡한 분기·반복 구조에서 히스토리 관리가 번거로움

→ LangGraph 방식(05_memory_langgraph.py)은 State 안에 messages를 두어 이 문제를 해결.
''')


## 결과를 검토하기

- 예상 동작: 같은 세션은 이전 내용을 이어 받고 다른 세션은 분리된다.
- 확인 질문: 같은 세션 ID를 재사용하면 어떤 기록이 보이나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
